export HUGGING_FACE_HUB_TOKEN=hf_

In [1]:
import torch
from sentence_transformers import SentenceTransformer

#device = "cuda" if torch.cuda.is_available() else "cpu"

model_id = "google/embeddinggemma-300M"
model = SentenceTransformer(model_id).to(device="cpu")

print(f"Device: {model.device}")
print(model)
print("Total number of parameters in the model:", sum([p.numel() for _, p in model.named_parameters()]))



/home/user/.conda/envs/sbert-env/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Device: cpu
SentenceTransformer(
  (0): Transformer({'max_seq_length': 2048, 'do_lower_case': False, 'architecture': 'Gemma3TextModel'})
  (1): Pooling({'word_embedding_dimension': 768, 'pooling_mode_cls_token': False, 'pooling_mode_mean_tokens': True, 'pooling_mode_max_tokens': False, 'pooling_mode_mean_sqrt_len_tokens': False, 'pooling_mode_weightedmean_tokens': False, 'pooling_mode_lasttoken': False, 'include_prompt': True})
  (2): Dense({'in_features': 768, 'out_features': 3072, 'bias': False, 'activation_function': 'torch.nn.modules.linear.Identity'})
  (3): Dense({'in_features': 3072, 'out_features': 768, 'bias': False, 'activation_function': 'torch.nn.modules.linear.Identity'})
  (4): Normalize()
)
Total number of parameters in the model: 307581696


In [2]:
# Run inference with queries and documents
query = "Which planet is known as the Red Planet?"
documents = [
    "Venus is often called Earth's twin because of its similar size and proximity.",
    "Mars, known for its reddish appearance, is often referred to as the Red Planet.",
    "Jupiter, the largest planet in our solar system, has a prominent red spot.",
    "Saturn, famous for its rings, is sometimes mistaken for the Red Planet."
]
query_embeddings = model.encode_query(query)
document_embeddings = model.encode_document(documents)
print(query_embeddings.shape, document_embeddings.shape)
# (768,) (4, 768)

# Compute similarities to determine a ranking
similarities = model.similarity(query_embeddings, document_embeddings)
print(similarities)

(768,) (4, 768)
tensor([[0.3008, 0.6361, 0.4927, 0.4889]])


In [11]:
text = """Giorgia Meloni has condemned the boss of Italy’s biggest trade union after he referred to the prime minister as the “courtesan” of Donald Trump.

Maurizio Landini, the leader of CGIL, which organised several pro-Palestinian protests before the Gaza ceasefire deal, made the remarks on TV on Tuesday, the day after world leaders, including Meloni, met in Egypt for a Middle East peace summit.

Landini accused Meloni of “not having lifted a finger” to bring peace in Gaza, limiting her role to “playing Trump’s courtesan”. “Fortunately, the Italian citizens took to the street to defend the dignity and honour of this country,” he said.

In a post on social media on Thursday, Meloni said Landini was “evidently clouded by a mounting resentment (which I can understand)”, before sharing a definition of “courtesan”.

“I think everyone knows the most common meaning attributed to this word, but, for the benefit of those who might not, I’m publishing the first definition found through a quick internet search,” Meloni said, posting a screenshot that read: “Woman of easy virtue, heterosexual; euphemism, prostitute.”

Meloni also criticised her leftwing opponents, saying that for decades they had “lectured us on respect for women” only to then criticise a woman by “calling her a prostitute”.

In response to the post, Landini argued there were “no sexist insults” towards Meloni and that he had used the term to imply “Trump’s lackey”.

In a statement, he said: “In a 10-minute interview, which anyone can easily rewatch, to avoid any misunderstanding or exploitation of the term used, I immediately clarified what I meant – that Meloni was on the coat tails of Trump, she was at Trump’s court, she was Trump’s lackey.”

Meloni has long sought to nurture friendly relations with the US president. During a speech in Egypt, Trump turned to Meloni – the only woman at the event – and called her “beautiful”. “In the United States, it would be the end of your political career,” he said. “But I’ll take the risk. Do you mind if I say you’re beautiful? Because you truly are beautiful.”

The US president later praised her as “an inspiration to all” in reference to the English version of her book, I Am Giorgia: My Roots, My Principles, which has a foreword by his son Donald Trump Jr.

In August, Meloni also hit back after discovering that doctored photos of her and other prominent Italian women had been posted on a pornographic website, saying she was “disgusted” and expressing “solidarity and support to all the women who have been offended, insulted and violated”."""


query = "donal trump meloni seism criticism palastinian"
random = "I am the captain of everything. which includes italians"
documents = [text, random]

In [ ]:
query_embeddings = model.encode_query(query)
document_embeddings = model.encode_document(documents)
print(query_embeddings.shape, document_embeddings.shape)
# (768,) (4, 768)

# Compute similarities to determine a ranking
similarities = model.similarity(query_embeddings, document_embeddings)
print(similarities)

(768,) (2, 768)
tensor([[0.5424, 0.1606]])


In [13]:
from pydantic import BaseModel, Field


class Summary(BaseModel):
    summary: str = Field(
        description="summary"

    )
    confidence: float = Field(
        description="the confidence of the summarised text"
    )

```python
summarization_params = {
    "top_k": 50,  # Top 50 candidates for the next token to ensure a balance of diversity and relevance in summaries
    "top_p": 0.95,  # Nucleus sampling probability, focusing on the top 95% of probability mass for more coherent summaries
    "max_tokens": 150,  # Maximum tokens for the summary (a concise summary, but not too short)
    "temperature": 0.7,  # Moderate temperature, keeping summaries focused but still creative
    "repeat_penalty": 1.2,  # A slight penalty to prevent exact repetition of phrases in the summary
    "frequency_penalty": 1.0,  # A moderate penalty to reduce the repetition of specific words/phrases
    "typical_p": 0.8,  # Makes the summary more likely to reflect common and sensible sentence structures
    "num_thread": 4,  # Use 4 threads for parallel processing—balances speed and system load
    "min_length": 50,  # Minimum length of the summary to ensure some detail is kept (but not too detailed)
    "max_length": 200,  # Maximum length to keep summaries concise and avoid unnecessary verbosity
    "length_penalty": 1.0,  # Neutral length penalty to allow the model to naturally generate summaries with varied lengths
    "use_attention": True,  # Enable attention mechanisms to focus on important sections, especially for longer documents
    "no_repeat_ngram_size": 3,  # Prevents repeating any sequence of 3 words or more to avoid redundancy
    "early_stopping": True,  # Stop generation once the summary reaches max length
}
```

In [21]:
options = {
    "top_k": 50,  # Top 50 candidates for the next token to ensure a balance of diversity and relevance in summaries
    "top_p": 0.95,  # Nucleus sampling probability, focusing on the top 95% of probability mass for more coherent summaries
    "max_tokens": 150,  # Maximum tokens for the summary (a concise summary, but not too short)
    "temperature": 0.7,  # Moderate temperature, keeping summaries focused but still creative
    "repeat_penalty": 1.2,  # A slight penalty to prevent exact repetition of phrases in the summary
    "frequency_penalty": 1.0,  # A moderate penalty to reduce the repetition of specific words/phrases
    "typical_p": 0.8,  # Makes the summary more likely to reflect common and sensible sentence structures
    "num_thread": 16,  # Use 4 threads for parallel processing—balances speed and system load
    "min_length": 50,  # Minimum length of the summary to ensure some detail is kept (but not too detailed)
    "max_length": 200,  # Maximum length to keep summaries concise and avoid unnecessary verbosity
    "length_penalty": 1.0,  # Neutral length penalty to allow the model to naturally generate summaries with varied lengths
    "use_attention": True,  # Enable attention mechanisms to focus on important sections, especially for longer documents
    "no_repeat_ngram_size": 3,  # Prevents repeating any sequence of 3 words or more to avoid redundancy
    "early_stopping": True,  # Stop generation once the summary reaches max length
}

In [18]:
from ollama import Client

host = "http://localhost:11434"
client = Client(host=host)

In [22]:
response = client.generate(
                                prompt=f"summarise the following: {text}",
                                system="you are a media analysis expert. Summarise the text to capture important embeddings",
                                model="gemma3:latest",
                                format=Summary.model_json_schema(),
                                options = options)

summary_text = Summary.model_validate_json(response.response)
summary_text

Summary(summary='This article details a heated exchange between Italian Prime Minister Giorgia Meloni and the leader of Italy’s largest trade union, Maurizio Landini, stemming from Landini’s inflammatory characterization of Meloni as “Trump’s courtesan.” The core of the conflict centers on Meloni’s relationship with Donald Trump and her role in the recent Middle East peace summit. Landini accused Meloni of inaction regarding Gaza and leveraging Trump’s influence. Meloni responded by aggressively defining “courtesan” and criticizing her left-wing opponents for past hypocrisy regarding women’s rights. The situation is further complicated by Trump’s public praise of Meloni and a prior incident involving doctored photos appearing on a pornographic website, which Meloni vehemently condemned. \n\n**Key Embeddings:**\n\n*   **Political Polarization:** The article highlights deep political divisions within Italy, particularly between Meloni’s right-wing government and the traditionally left-le

In [23]:
summary_text.summary

'This article details a heated exchange between Italian Prime Minister Giorgia Meloni and the leader of Italy’s largest trade union, Maurizio Landini, stemming from Landini’s inflammatory characterization of Meloni as “Trump’s courtesan.” The core of the conflict centers on Meloni’s relationship with Donald Trump and her role in the recent Middle East peace summit. Landini accused Meloni of inaction regarding Gaza and leveraging Trump’s influence. Meloni responded by aggressively defining “courtesan” and criticizing her left-wing opponents for past hypocrisy regarding women’s rights. The situation is further complicated by Trump’s public praise of Meloni and a prior incident involving doctored photos appearing on a pornographic website, which Meloni vehemently condemned. \n\n**Key Embeddings:**\n\n*   **Political Polarization:** The article highlights deep political divisions within Italy, particularly between Meloni’s right-wing government and the traditionally left-leaning trade unio

In [ ]:
from pydantic import BaseModel, Field


class Paraphrase(BaseModel):
    paraphrases: list[str] = Field(
        description="paraphrases"

    )
    confidences: list[float] = Field(
        description="the confidences of the summarised text"
    )
response = client.generate(
                                prompt=f"paraphrase the following for embeddings: {text}",
                                system="you are a media analysis expert. Your task is to paraphrase capturing the most important details to embed.",
                                model="gemma3:latest",
                                format=Paraphrase.model_json_schema(),
                                options = options)

paraphrases = Paraphrase.model_validate_json(response.response)
paraphrases

Paraphrase(paraphrases=['Following a contentious TV interview, Giorgia Meloni accused CGIL leader Maurizio Landini of portraying her as Donald Trump’s ‘courtesan,’ sparking a heated exchange.', 'Meloni’s response to Landini’s criticism included a direct definition of ‘courtesan’ and a critique of Italy’s left-wing opposition for past hypocrisy regarding women’s respect.', 'The conflict stems from Meloni’s efforts to cultivate a relationship with Donald Trump, who publicly praised her appearance and described her as an ‘inspiration,’ while Landini argues the ‘courtesan’ label accurately reflects her dependence on Trump’s influence.', 'The situation highlights a clash between Meloni’s right-wing politics and Landini’s pro-Palestinian activism, further complicated by a previous controversy involving doctored images and a connection to Donald Trump Jr.'], confidences=[0.85, 0.9, 0.75, 0.8], embeddings=[[4473.0, 4479.0, 4489.0, 4497.0, 4506.0], [4473.0, 4479.0, 4506.0], [4473.0, 4479.0, 450

In [27]:
paraphrase_embeddings = model.encode_document(paraphrases.paraphrases)

In [30]:
combined_embedding = paraphrase_embeddings.sum()

In [31]:
combined_embedding

np.float32(-7.7846622)

In [36]:
import numpy as np

def sum_embeddings(embeddings_list):
    """
    Sum embedding vectors element-wise across all paraphrases
    
    Args:
        embeddings_list: List of numpy arrays containing embeddings
        
    Returns:
        Combined embedding vector (numpy array)
    """
    # Verify all embeddings have same dimensions
    dims = embeddings_list[0].shape
    if not all(emb.shape == dims for emb in embeddings_list):
        raise ValueError("All embeddings must have same dimensions")
        
    # Sum vectors element-wise
    combined = np.zeros(dims)
    for emb in embeddings_list:
        combined += emb
        
    return combined.astype(np.float32)

In [37]:
combined_embedding = sum_embeddings(paraphrase_embeddings)

In [38]:
combined_embedding

array([-2.45514512e-01,  1.07005805e-01, -1.25722706e-01, -2.16950942e-02,
        2.06930023e-02,  2.89532561e-02, -2.27697473e-02, -4.40476201e-02,
        1.68726951e-01, -2.69142948e-02, -2.80585438e-01,  4.71440703e-02,
       -3.80531326e-02,  5.47704920e-02, -2.20535338e-01, -1.58450864e-02,
        1.17745794e-01,  1.41755477e-01, -9.94058922e-02, -1.85640872e-01,
        9.56116170e-02, -5.91580458e-02, -3.46582890e-01,  9.14738327e-02,
        2.21395027e-03,  2.33042747e-01, -2.55457126e-02, -8.41148570e-02,
        1.54414117e-01, -1.68800447e-02,  1.36547923e-01,  3.54745202e-02,
        1.93219468e-01, -2.46450707e-01,  1.25637442e-01,  3.67291830e-02,
        7.46352673e-02,  2.39561573e-02, -1.06003247e-01, -2.17256919e-01,
       -2.10012928e-01,  1.47687227e-01,  2.29282171e-01, -2.16017105e-02,
       -2.67220810e-02, -4.49976977e-03, -8.00719038e-02,  1.78472996e-02,
        5.56179732e-02, -5.43655269e-02,  5.41969389e-02, -1.05588168e-01,
       -1.13114804e-01,  

In [71]:
extract = """This article details a heated exchange between Italian Prime Minister Giorgia Meloni and the leader of Italy’s largest trade union, Maurizio Landini, stemming from Landini’s inflammatory characterization of Meloni as “Trump’s courtesan.” The core of the conflict centers on Meloni’s relationship with Donald Trump and her role in the recent Middle East peace summit."""
original = """But I’ll take the risk. Do you mind if I say you’re beautiful? Because you truly are beautiful"""

query_emb = [
    model.encode_query("prostitute meloni"),
    model.encode_query("nasa space station"),
    model.encode_query("sexism"),
    model.encode_query("donald trump"),
    model.encode_query("aurizio Landini"),
    model.encode_query("respect for women"),
    model.encode_query("trade unions fight against meloni"),
    model.encode_query(extract),
    model.encode_query(original),
]
similarities = model.similarity(query_emb, combined_embedding)
print(similarities)

tensor([[0.5601],
        [0.0457],
        [0.2455],
        [0.3110],
        [0.4384],
        [0.2367],
        [0.5452],
        [0.7557],
        [0.1631]])


In [69]:
a = model.encode_query("Ruler of France")

b = model.encode_query("Leader of french people")

similarities = model.similarity(a, b)
print(similarities)

tensor([[0.6844]])


In [72]:
a = model.encode_query("Ruler of France")

b = model.encode_document("Leader of french people")

similarities = model.similarity(a, b)
print(similarities)

tensor([[0.5164]])


## Embedding
- Entity embedding (fine tuned model?)
- Read paper: encode query vs. document